In [1]:
import torch
import transformers
import datasets
import peft
import trl
import accelerate
import wandb

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("CUDA:", torch.cuda.is_available())
print("wandb:", wandb.__version__)

c:\Users\stron\anaconda3\envs\llm_clean\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


: 

In [1]:
import datasets

print(datasets.__version__)

c:\Users\stron\anaconda3\envs\llm_clean\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


4.8.5


# Huggingface login and WandB login

In [1]:
from dotenv import load_dotenv
from huggingface_hub import login
import wandb
import os

load_dotenv()

login(token=os.getenv("HF_TOKEN"))
wandb.login(key=os.getenv("WANDB_API_KEY"))


c:\Users\stron\anaconda3\envs\llm_clean\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\stron\_netrc
wandb: Currently logged in as: saba-rezaee-khavas (saba-rezaee-khavas-umass-lowell) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [2]:
run = wandb.init(
    project='Fine-tune Gemma-2-2b-it on doctor-healthcare dataset',
    job_type="training",
    anonymous="allow"
)

wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


# Loading Data

In [1]:
import pandas as pd
df = pd.read_csv("../data/Doctor-HealthCare-100k.csv")
print(df.head())

                                         instruction  \
0  If you are a doctor, please answer the medical...   
1  If you are a doctor, please answer the medical...   
2  If you are a doctor, please answer the medical...   
3  If you are a doctor, please answer the medical...   
4  If you are a doctor, please answer the medical...   

                                               input  \
0  I woke up this morning feeling the whole room ...   
1  My baby has been pooing 5-6 times a day for a ...   
2  Hello, My husband is taking Oxycodone due to a...   
3  lump under left nipple and stomach pain (male)...   
4  I have a 5 month old baby who is very congeste...   

                                              output  
0  Hi, Thank you for posting your query. The most...  
1  Hi... Thank you for consulting in Chat Doctor....  
2  Hello, and I hope I can help you today.First, ...  
3  HI. You have two different problems. The lump ...  
4  Thank you for using Chat Doctor. I would sugge..

In [2]:
def format_chatml(example):
    text = (
        "<|im_start|>system\n"
        f"{example['instruction']}"
        "<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['input']}"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['output']}"
        "<|im_end|>"
    )

    return {"text": text}


In [3]:
from datasets import Dataset # Import the Dataset class

# Convert pandas DataFrame to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df)

# Now map the format_example function to the Hugging Face Dataset
dataset = hf_dataset.map(format_chatml)


dataset['text'][1]

c:\Users\stron\anaconda3\envs\llm_clean\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 112156/112156 [00:03<00:00, 37368.53 examples/s]


"<|im_start|>system\nIf you are a doctor, please answer the medical questions based on the patient's description.<|im_end|>\n<|im_start|>user\nMy baby has been pooing 5-6 times a day for a week. In the last few days it has increased to 7 and they are very watery with green stringy bits in them. He does not seem unwell i.e no temperature and still eating. He now has a very bad nappy rash from the pooing ...help!<|im_end|>\n<|im_start|>assistant\nHi... Thank you for consulting in Chat Doctor. It seems your kid is having viral diarrhea. Once it starts it will take 5-7 days to completely get better. Unless the kids having low urine output or very dull or excessively sleepy or blood in motion or green bilious vomiting...you need not worry. There is no need to use antibiotics unless there is blood in the motion. Antibiotics might worsen if unnecessarily used causing antibiotic associated diarrhea. I suggest you use zinc supplements (Z&D Chat Doctor.<|im_end|>"

In [4]:
dataset = dataset.train_test_split(test_size=0.1)
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 100940
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 11216
    })
})

# Fine-tuning Steps for Gemma 2 Using LoRA On top of Qlora 4 bit Quantizattion

In [ ]:
base_model = "google/gemma-2-2b-it"
new_model = "Gemma-2-2b-it"

In [ ]:
# QLoRA config
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,   # this is the Q in QLORA, we load the model in 4-bit precision, 
    #Instead of loading Gemma weights in FP16 or BF16, we load them in 4-bit.
    bnb_4bit_quant_type="nf4",   # it is just a format for quantization, it is not a precision, it is a way to quantize the weights
    bnb_4bit_compute_dtype=torch.float16,     # The model is stored in 4-bit, but calculations are done in BF16.
    bnb_4bit_use_double_quant=True,
)


### Load Quantized Model

In [ ]:
# Load quantized model
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load Gemma in 4-bit

model = AutoModelForCausalLM.from_pretrained(  #CausalLM means the model predicts the next token.
    base_model,
    quantization_config=bnb_config,
    device_map="auto",     # automatically put the model on the GPU if available, otherwise on CPU
    attn_implementation="eager"    #This controls how attention is computed. for Gemma it is normally "eager"
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)


Loading weights: 100%|██████████| 288/288 [00:01<00:00, 173.68it/s]


### test model and tokenizer

In [18]:
input_text = "How are you today? how have you been recently?"
input_ids = tokenizer(input_text, return_tensors="pt").to("cuda")
outputs = model.generate(**input_ids, max_new_tokens=200)
print(outputs)
print(tokenizer.decode(outputs[0]))

tensor([[     2,   2299,    708,    692,   3646, 235336,   1368,    791,    692,
           1125,   7378, 235336,    109, 235285, 235303, 235262,   3900,   1578,
         235269,   7593,    692, 235341,    590, 235303,    524,   1125,  13572,
            675,   1160,    578,   1009,   3749,   7340, 235269,    901,   8691,
         235269,    590, 235303, 235262,   4915, 235265,   2250,   1105,    692,
         235336,  44416, 235248,    108,    107]], device='cuda:0')
<bos>How are you today? how have you been recently?

I'm doing well, thank you! I've been busy with work and some personal projects, but overall, I'm happy. How about you? 😊 
<end_of_turn>


In [19]:
model

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Gemma2TextScaledWordEmbedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear4bit(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear4bit(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)


In [ ]:
### The function traverses the quantized model, identifies all Linear4bit layers ( all the layer that has been 
# quantized when the model was loaded load_in_4bit=True) created by BitsAndBytes, extracts their module names, 
# and returns them as LoRA target modules. This allows PEFT to automatically inject LoRA adapters into the model's 
# attention and feed-forward projection layers without manually specifying each layer. some people actually hard 
# code that as modules = ['down_proj', 'o_proj', 'up_proj', 'v_proj', 'gate_proj', 'q_proj', 'k_proj']





import bitsandbytes as bnb

def find_all_linear_names(model):
    cls = bnb.nn.Linear4bit
    lora_module_names = set()
    for name, module in model.named_modules():
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])
    if 'lm_head' in lora_module_names:  # needed for 16 bit
        lora_module_names.remove('lm_head')
    return list(lora_module_names)

modules = find_all_linear_names(model)

In [25]:
print(modules)

['up_proj', 'q_proj', 'v_proj', 'o_proj', 'down_proj', 'k_proj', 'gate_proj']


In [ ]:
# LoRA config LoRA adds small, low-rank matrices to each layer, allowing only these matrices to be trained.
#This minimizes the computational load and memory needed.


from peft import (
    LoraConfig,
    get_peft_model,            # injects lora adapters to the model
)



tokenizer.chat_template = None      # this disables any built-in chat formatting
peft_config = LoraConfig(
    r=16,                           #  the rank in LoRA directly affects the number of trainable parameters in the model. 
    lora_alpha=32,              # Roughly controls how strongly the adapter influences the model.
    lora_dropout=0.05,
    bias="none",                     # dont train biases, only train lora adapters
    task_type="CAUSAL_LM",
    target_modules=modules,
)

model = get_peft_model(model, peft_config)

c:\Users\stron\anaconda3\envs\llm_clean\lib\site-packages\peft\mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
c:\Users\stron\anaconda3\envs\llm_clean\lib\site-packages\peft\tuners\tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [29]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): PeftModelForCausalLM(
      (base_model): LoraModel(
        (model): Gemma2ForCausalLM(
          (model): Gemma2Model(
            (embed_tokens): Gemma2TextScaledWordEmbedding(256000, 2304, padding_idx=0)
            (layers): ModuleList(
              (0-25): 26 x Gemma2DecoderLayer(
                (self_attn): Gemma2Attention(
                  (q_proj): lora.Linear4bit(
                    (base_layer): Linear4bit(in_features=2304, out_features=2048, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=2304, out_features=16, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=16, out_features=2048, bias=False)
                    )
                    (lora_em

# Training

In [ ]:

from transformers import TrainingArguments

training_arguments = TrainingArguments(
    output_dir=new_model,
    per_device_train_batch_size=1,  #process 1 example at a time,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,   #but update weights after 2 training examples
    optim="paged_adamw_32bit",
    num_train_epochs=1,
    eval_strategy="steps",
    eval_steps=0.2,   # evaluate every 20% of training
    logging_steps=1,   # Print training metrics every step
    warmup_steps=10,
    logging_strategy="steps",
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    report_to="wandb"
)

# Setting sft parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=training_arguments,
)

model.config.use_cache = False
trainer.train()

Tokenizing eval dataset: 100%|██████████| 11216/11216 [00:04<00:00, 2768.30 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
10094,2.238945,1.936818,1.956838,5759367.000000,0.584328
20188,2.287543,1.875842,1.887090,11551050.000000,0.594285
30282,1.840909,1.832501,1.822124,17332766.000000,0.601112
40376,1.327758,1.803501,1.812311,23135086.000000,0.606039
50470,1.724057,1.796705,1.795332,28928259.000000,0.607117


TrainOutput(global_step=50470, training_loss=1.8965417022820343, metrics={'train_runtime': 126017.9821, 'train_samples_per_second': 0.801, 'train_steps_per_second': 0.4, 'total_flos': 3.549991372136248e+17, 'train_loss': 1.8965417022820343, 'epoch': 1.0})

In [ ]:
training_arguments = TrainingArguments(
    output_dir=new_model,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,

    optim="paged_adamw_32bit",
    num_train_epochs=1,

    eval_strategy="steps",
    eval_steps=50,           # Evaluate every 50 steps. This should match save_steps when using best-model tracking.

    save_strategy="steps",
    save_steps=50,          #Save a checkpoint every 50 optimizer update steps.

    load_best_model_at_end=True,        # reload the best checkpoint at the end of the training
    metric_for_best_model="eval_loss",   # use eval_loss to evaluate the best model, the lower the better
    greater_is_better=False,

    save_total_limit=2,   # keep only best + most recent checkpoints

    logging_steps=1,
    warmup_steps=10,
    logging_strategy="steps",
    learning_rate=2e-4,

    fp16=False,
    bf16=False,
    report_to="wandb",
)




trainer.train()
trainer.save_model("./best_gemma_qlora_model")

# load the best model

In [1]:
best_checkpoint = "Gemma-2-2b-it\checkpoint-50470"
base_model = "google/gemma-2-2b-it"

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

best_checkpoint = "../Gemma-2-2b-it/checkpoint-50470"
base_model = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(base_model)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
)

model = PeftModel.from_pretrained(base, best_checkpoint)
model.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 288/288 [00:02<00:00, 131.69it/s]
c:\Users\stron\anaconda3\envs\llm_clean\lib\site-packages\peft\peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma2ForCausalLM(
      (model): Gemma2Model(
        (embed_tokens): Gemma2TextScaledWordEmbedding(256000, 2304, padding_idx=0)
        (layers): ModuleList(
          (0-25): 26 x Gemma2DecoderLayer(
            (self_attn): Gemma2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2304, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2304, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
    

In [9]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
    )

tokenized_eval = dataset["test"].map(tokenize, batched=True, remove_columns=dataset["test"].column_names)

eval_args = TrainingArguments(
    output_dir="./eval_results",
    per_device_eval_batch_size=1,
    report_to="none",
)

trainer = Trainer(
    model=model,
    eval_dataset=tokenized_eval,
    args=eval_args,
    data_collator=DataCollatorWithPadding(tokenizer),
)

metrics = trainer.evaluate()
print(metrics)


Map: 100%|██████████| 11216/11216 [00:00<00:00, 12930.96 examples/s]


Training Loss,Validation Loss,Step
No log,No log,0


{}
